In [ ]:
import numpy as np
import csv, h5py
from numpy import linalg
import os, sys, scipy
path = "/home/jung/Dokumente/projects/nonclassicality/InterpretableStatCorrModel"
os.chdir(path) # change path to parent directory
sys.path.append(path) # Add the parent directory to sys.path

import matplotlib.pyplot as plt
from tqdm import tqdm

plt.rcParams['text.usetex'] = True
plt.rcParams['font.size'] = 20
plt.rcParams['font.family'] = "serif"

## Helper functions

In [ ]:
def load_h5py_ds(dataset, kshots):
    data = h5py.File(f'saved_params/{dataset}/train_test_{dataset}_M{1000*kshots}.hdf5', 'r')
    train_images = np.array(data["train_ds/images"])
    train_labels = np.array(data["train_ds/labels"])
    data.close()
    return {'images': train_images, 'labels': train_labels}

def TraditionalPrediction(ds, trad_witness, kshots, modes, N_detectors,
                        tolerance_min, tolerance_max, step_size, SAVENAME):
    if trad_witness not in [BinomialParameter, Order3Clickwitness, GeneralizedKlyshko_click_witness]:
        for t in tqdm(range(tolerance_min, tolerance_max+1)):
            cl, ncl = trad_witness(ds, t*step_size, kshots, modes)
            # Create csv file
            with open(f"{SAVENAME}.csv", "a") as f:
                writer = csv.writer(f)
                writer.writerow([round(t*step_size,4), round(cl,4),round(ncl,4)])
    elif trad_witness in [BinomialParameter, Order3Clickwitness,GeneralizedKlyshko_click_witness]:
        for t in tqdm(range(tolerance_min, tolerance_max+1)):
            cl, ncl = trad_witness(ds, N_detectors, t*step_size, kshots, modes)
            # Create csv file
            with open(f"{SAVENAME}.csv", "a") as f:
                writer = csv.writer(f)
                writer.writerow([round(t*step_size,2), round(cl,4), round(ncl,4)])

## PNR statistics

#### $2^{nd}$-order witness: $Q_B$
The Mandel-Q parameter quantifies the Poissonian-nature of a photon number distribution. It is defined as
$$ Q_M =\frac{\langle \hat{n}^2\rangle - \langle \hat{n}\rangle^2}{\langle\hat{n}\rangle}-1 $$
where $\langle \hat{n}^2\rangle$ is the second moment of the photon-number operator.

#### $3^{rd}$-order moment-based witness
Krishnaswamy et al. developed a generalization of this criterion by considering half-integer labeling in the construction of the matrix of moments. The smallest minor then gives the following criterion:
$$ Q_3 := \langle\hat{n}\rangle\langle:\hat{n}^3:\rangle - \langle:\hat{n}^2:\rangle^2 \overset{\mathrm{cl.}}{\geq}0  $$
This can be simplified to
$$ Q_3 = \langle \hat{n}\rangle\langle\hat{n}^3\rangle -\langle\hat{n}^2 \rangle^2 +  \langle \hat{n}^2\rangle\langle\hat{n}\rangle + \langle\hat{n}\rangle^2$$

In [3]:
def MandelQ(ds, tolerance, kshots, modes):
    M= 1000*kshots
    Noncls_predictions = []
    Cls_predictions = []
    for qstate in range(len(ds['labels'])):
        Q = []
        x = ds['images'][qstate]
        for j in range(modes):
            n = x[j,:]
            Q.append( (np.sum(n**2)/M-(np.sum(n)/M)**2)/(np.mean(n))-1)
        prediction = float(min(Q)+tolerance < 0.0)
        if ds['labels'][qstate] == 0.0:
            Cls_predictions.append(prediction)
        elif ds['labels'][qstate] == 1.0:
            Noncls_predictions.append(prediction)
    Cls_predictions = np.array(Cls_predictions)
    Noncls_predictions = np.array(Noncls_predictions)
    Cls_accuracy = round(1-np.sum(Cls_predictions)/len(Cls_predictions),4)
    Ncl_accuracy = round(np.sum(Noncls_predictions)/len(Noncls_predictions),4)
    return Cls_accuracy, Ncl_accuracy

def Order3witness(ds, tolerance, kshots, modes):
    M= 1000*kshots
    Noncls_predictions = []
    Cls_predictions = []
    for qstate in range(len(ds['labels'])):
        Q_3 = []
        x = ds['images'][qstate]
        for j in range(modes):
            n = x[j,:]
            tmp1 = np.mean(n)*np.mean(n**3) - np.mean(n)*np.mean(n**2)
            tmp2 = np.mean(n)**2 -(np.mean(n**2))**2
            Q_3.append(tmp1+tmp2)
        prediction = float(min(Q_3)+tolerance < 0.0)
        if ds['labels'][qstate] == 0.0:
            Cls_predictions.append(prediction)
        elif ds['labels'][qstate] == 1.0:
            Noncls_predictions.append(prediction)
    Cls_predictions = np.array(Cls_predictions)
    Noncls_predictions = np.array(Noncls_predictions)
    Cls_accuracy = round(1-np.sum(Cls_predictions)/len(Cls_predictions),4)
    Ncl_accuracy = round(np.sum(Noncls_predictions)/len(Noncls_predictions),4)
    return Cls_accuracy, Ncl_accuracy

def Order4witness(ds, tolerance, kshots, modes):
    M=1000*kshots
    Noncls_predictions = []
    Cls_predictions = []
    for qstate in range(len(ds['labels'])):
        Q = []
        x = ds['images'][qstate]
        for j in range(modes):
            n = x[j,:]
            m1 = np.mean(n)
            m2 = np.sum(n**2-n)/M
            m3 = np.sum(n**3-3*n**2+2*n)/M
            m4 = np.sum(n**4-6*n**3+11*n**2-6*n)/M
            Q.append(m2*m4 + 2*m1*m2*m3 -m2**3-m3**2-m1**2*m4)
        prediction = float(min(Q)+tolerance < 0.0)
        if ds['labels'][qstate] == 0.0:
            Cls_predictions.append(prediction)
        elif ds['labels'][qstate] == 1.0:
            Noncls_predictions.append(prediction)
    Cls_predictions = np.array(Cls_predictions)
    Noncls_predictions = np.array(Noncls_predictions)
    Cls_accuracy = round(1-np.sum(Cls_predictions)/len(Cls_predictions),4)
    Ncl_accuracy = round(np.sum(Noncls_predictions)/len(Noncls_predictions),4)
    return Cls_accuracy, Ncl_accuracy

### $2^{nd}$-order witness for 6-mode system

The 6-mode dataset is benchmarked with the second-order matrix of moments, generalized to 6-modes. For this purpose, we investigate the following matrix:

![image](6_mode_witness.png)

For all classical states $M\overset{\mathrm{cl.}}{\geq} 0$, hence, we have to find the minimal eigenvalue of $M$ and check its positivity.

In [4]:
def Second_order_6_mode(ds, tolerance, kshots, modes):
    M = np.zeros((modes+1,modes+1))
    shots=1000*kshots
    Noncls_predictions = []
    Cls_predictions = []
    for qstate in range(len(ds['labels'])):
        x = ds['images'][qstate]
        first_row = [np.mean(x[modes-(j+1),:]) for j in range(modes)]
        M[1:,0] = first_row
        for j in range(modes-1,-1,-1):
            for k in range(modes-1,j,-1):
                n_j = x[j,:]
                n_k = x[k,:]
                M[modes-j,modes-k] = np.mean(n_j*n_k)
        # fill upper triagle
        M = M + M.T
        # fill diagonal
        M[0,0] = 1.
        for l in range(modes):
            n_l = x[modes-1-l]
            M[1+l,1+l] = np.mean(n_l**2)
        # compute minimal eigenvalue
        min_eigenval = np.min(linalg.eigvals(M))
        prediction = float(min_eigenval+tolerance < 0.0)
        if ds['labels'][qstate] == 0.0:
            Cls_predictions.append(prediction)
        elif ds['labels'][qstate] == 1.0:
            Noncls_predictions.append(prediction)
    Cls_predictions = np.array(Cls_predictions)
    Noncls_predictions = np.array(Noncls_predictions)
    Cls_accuracy = round(1-np.sum(Cls_predictions)/len(Cls_predictions),4)
    Ncl_accuracy = round(np.sum(Noncls_predictions)/len(Noncls_predictions),4)
    return Cls_accuracy, Ncl_accuracy

# # Test function
# x = np.array([[.1*i for i in range(1,7)]]*100).T
# print('example x =',x[:,0])
# ds = {'images': np.array([x]), 'labels': np.array([0])}
# cl, ncl = Second_order_6_mode(ds, tolerance=0,kshots=100, modes=6)

# Click statistics
#### $2^{nd}$-order witness: $Q_B$
In the case of click statistics, the Mandel-Q parameter is replaced by the Binomial parameter
$ Q_B \propto \langle :\hat{\pi}^2:\rangle - \langle:\hat{\pi}:\rangle^2 \overset{\mathrm{cl.}}{\geq} 0$
This can be rewritten to take the form
$$ Q_B \propto \langle c^2\rangle - \frac{N-1}{N}\langle c\rangle^2 - \langle c\rangle$$
where $N$ is the total number of bins in the detection scheme.

#### $3^{rd}$ order witness
This witness can be extended to third order by expanding the moments in the matrixof moments with half-integers. The lowest minor then yields the following criterion:
$Q_{B,3} := \langle\hat{\pi}\rangle \langle :\hat{\pi}^3:\rangle - \langle:\hat{\pi}^2:\rangle^2 \overset{cl.}{\geq}0 $. Using the generating function as outlines in https://link.aps.org/doi/10.1103/PhysRevLett.109.093601
$$ f(x) = \sum_k c_k x^k  = \langle :\left[x(\mathbb{1} - e^{-\hat{n}/N}) + e^{-\hat{n}/N} \right]^N :\rangle,$$
the normal ordered moments of the click operator $\mathbb{1}-e^{-\hat{n}/N}$ can be derived by evaluating derivatives of the generating function at the point $x=1$. The third order witness can then be expressed as
$$Q_{B,3}= \frac{\langle c^3\rangle - 3\langle c^2\rangle + 2\langle c\rangle}{N^2(N-1)(N-2)} - \left( \frac{\langle c^2\rangle - \langle c\rangle}{N(N-1)} \right)^2 $$
By multiplying with $N(N-1)(N-2)$ this expression can be rewritten as
$$ Q_{B,3} \propto \langle \hat{c}^3\rangle\langle \hat{c}\rangle - 3\langle \hat{c}^2\rangle\langle\hat{c}\rangle + 2\langle \hat{c}\rangle^2 - \frac{N-2}{N-1} \left[ \langle \hat{c}^2\rangle^2 - 2\langle \hat{c}^2\rangle\langle\hat{c}\rangle + \langle \hat{c}\rangle^2 \right]$$

In [ ]:
def BinomialParameter(ds, N_detectors,tolerance, kshots, modes):
    ''' 
        Implements the binomial parameter Q_B = N var(c)²/(<c>(N-<c>))-1
    '''
    M= 1000*kshots
    Noncls_predictions = []
    Cls_predictions = []
    for qstate in range(len(ds['labels'])):
        Q = []
        x = ds['images'][qstate]
        for j in range(modes):
            c = x[j,:]
            nom = np.var(c)
            c_mean = np.mean(c)
            #denom = N_detectors*np.mean(c)-np.mean(c)**2
            denom = 1
            Q.append( nom - c_mean + (c_mean**2)/(N_detectors))
            #Q.append( N_detectors*nom/denom -1)
        prediction = float(min(Q)+tolerance < 0.0)
        if ds['labels'][qstate] == 0.0:
            Cls_predictions.append(prediction)
        elif ds['labels'][qstate] == 1.0:
            Noncls_predictions.append(prediction)
    Cls_predictions = np.array(Cls_predictions)
    Noncls_predictions = np.array(Noncls_predictions)
    Cls_accuracy = round(1-np.sum(Cls_predictions)/len(Cls_predictions),4)
    Ncl_accuracy = round(np.sum(Noncls_predictions)/len(Noncls_predictions),4)
    return Cls_accuracy, Ncl_accuracy

def Order3Clickwitness(ds, N_detectors,tolerance, kshots, modes):
    M= 1000*kshots
    Noncls_predictions = []
    Cls_predictions = []
    for qstate in range(len(ds['labels'])):
        Q = []
        x = ds['images'][qstate]
        for j in range(modes):
            c = x[j,:]
            tmp1 = (np.mean(c**3) - 3*np.mean(c**2) + 2*np.mean(c))*np.mean(c)
            tmp2 = np.mean(c**2)**2 - 2*np.mean(c**2)*np.mean(c) + np.mean(c)**2
            N_factor = (N_detectors-2)/(N_detectors-1)
            Q.append(tmp1 - N_factor*tmp2 )
        prediction = float(min(Q)+tolerance < 0.0)
        if ds['labels'][qstate] == 0.0:
            Cls_predictions.append(prediction)
        elif ds['labels'][qstate] == 1.0:
            Noncls_predictions.append(prediction)
    Cls_predictions = np.array(Cls_predictions)
    Noncls_predictions = np.array(Noncls_predictions)
    Cls_accuracy = round(1-np.sum(Cls_predictions)/len(Cls_predictions),4)
    Ncl_accuracy = round(np.sum(Noncls_predictions)/len(Noncls_predictions),4)
    return Cls_accuracy, Ncl_accuracy

#### Probability-based witness: Generalized Klyshko
Consider a multiplexing detection scheme with $N$ total bins. The POVM of no clicks is given by $\hat{\pi} = e^{-\eta\hat{n}/N}$ where $\hat{n}$ is the photon-number operator and $\eta$ the efficiency of the detector. The click counting distribution for $k$ clicks is determined by $\hat{\Pi}_k = \binom{N}{k}:(1-\hat{\pi})^k(\hat{\pi})^{N-k}:$.

A computation simular to the one for photon-number statistics with $\hat{f}=\sum_{k=0}^{\lfloor \frac{N}{2}\rfloor} f_k \hat{\pi}^{-N/2} \frac{\hat{\Pi}_k}{\binom{N}{k}}$ yields

$$ 0 \overset{\mathrm{cl.}}{\leq} M_{j,k} = \left( \frac{c_{k+j}}{\binom{N}{j+k}} \right)_{j,k} $$

In [ ]:
def MinEigenval_click_int(rho, N_detectors, cutoff):  
    if cutoff < 3:
        #print('returning 1.0 since Matrix is just a number')
        return 1.0
    else:
        M = np.zeros((int((cutoff+1)/2),int((cutoff+1)/2)))
        for j in range(int((cutoff+1)/2)):
            for k in range(int((cutoff+1)/2)):
                M[j,k] = rho[j+k]/scipy.special.binom(N_detectors,j+k)
        min_eigenval = np.min(linalg.eigvals(M))
        return min_eigenval

def MinEigenval_click_half_int(rho, N_detectors,cutoff):  
    if cutoff < 4:
        return 1.0
    else:
        M = np.zeros((int(cutoff/2),int(cutoff/2)))
        for j in np.arange(0.5, int(cutoff/2)):
            for k in np.arange(0.5, int(cutoff/2)):
                #print('filling index ', int(j-0.5), int(2*k-1))
                M[int(j-0.5),int(k-0.5)] = rho[int(j+k)]/scipy.special.binom(N_detectors,j+k)
        min_eigenval = np.min(linalg.eigvals(M))
        return min_eigenval

def GeneralizedKlyshko_click_witness(ds, N_detectors, tolerance, kshots, modes):
    list_eigenvals_int = []
    list_eigenvals_half_int = []
    Cls_predictions = []
    Noncls_predictions = []
    for qstate in range(len(ds['labels'])):
        D_ks = []
        x = ds['images'][qstate]
        for j in range(modes):
            c = x[j,:]
            max_photon_number = np.max(c)
            p_tmp, bin = np.histogram(c, bins = np.arange(int(max_photon_number+2)), density=True)

            # compute minmal eigenvalue of matrix
            min_eigenval_int = MinEigenval_click_int(rho=p_tmp, 
                                            N_detectors=N_detectors,
                                            cutoff=max_photon_number)
            min_eigenval_half_int = MinEigenval_click_half_int(rho=p_tmp, 
                                                            N_detectors=N_detectors,
                                                            cutoff=max_photon_number)
            list_eigenvals_int.append(min_eigenval_int)
            list_eigenvals_half_int.append(min_eigenval_half_int)

            if ds['labels'][qstate] == 0.0:
                Cls_predictions.append(min(min_eigenval_int, min_eigenval_half_int) <-1.*tolerance)
            elif ds['labels'][qstate] == 1.0:
                Noncls_predictions.append(min(min_eigenval_int, min_eigenval_half_int)<-1.*tolerance)

    # compute accuracy
    Cls_predictions = np.array(Cls_predictions)
    Noncls_predictions = np.array(Noncls_predictions)
    Cls_accuracy = round(1-np.sum(Cls_predictions)/len(Cls_predictions),4)
    Ncl_accuracy = round(np.sum(Noncls_predictions)/len(Noncls_predictions),4)
    return Cls_accuracy, Ncl_accuracy

## Truncated photon-number statistics

### Klyshko
For $k\geq 1$ the following inequality is a sufficient criterion for nonclassicality
$$ kp_k^2 > (k+1)p_{k+1}p_{k-1}$$
where $p_k$ is the probability for observing $k$ photons.


### Innocenti (not used in the paper)
Defining $Q_k = k! p_k$, Innocenti developed a witness that takes into account the tail of the photon-number distribution. All classical states staisfy the following inequality
$$ \sum_{k=0}^{N-2} p_k + \frac{Q_{N-1}^N}{Q_N^{N-1}} \left[ e^{Q_N/Q_{N-1}} \sum_{k=0}^{N-2} \frac{(Q_N/Q_{N-1})^k}{k!} \right] \overset{\mathrm{cl.}}{\leq }1$$
where $N$ is the maximal number of photons measured.

In [ ]:
def Klyshkos_witness(ds, epsilon, kshots, modes):
    M = 1000*kshots
    Noncls_predictions = []
    Cls_predictions = []
    count = 0
    for qstate in range(len(ds['labels'])):
        D_ks = []
        x = ds['images'][qstate]
        for j in range(modes):
            n = x[j,:]
            D_k_mode = []
            p_tmp, bin = np.histogram(n, bins = np.arange(int(np.max(n)+2)), density=True)
            if np.max(n) == 0:
                D_k_mode.append(1.0)
                D_ks.append(D_k_mode)
            elif np.max(n) == 1:
                # modified Klyshko criterion
                D_k_mode.append(np.sqrt(p_tmp[0]*(2-p_tmp[0])) -p_tmp[0] - p_tmp[1])
                D_ks.append(D_k_mode)
            elif np.max(n) == 2:
                D_k_mode.append(2*p_tmp[0]*p_tmp[2] - p_tmp[1]**2)
                D_ks.append(D_k_mode)
            else:
                for k in range(int(np.max(n))-2):
                    D_k_mode.append((k+2)*p_tmp[k]*p_tmp[k+2]-(k+1)*(p_tmp[k+1]**2))
                D_ks.append(np.min(D_k_mode))
        prediction = float(np.min(D_ks) < (0.0 - epsilon) )
        if ds['labels'][qstate] == 0.0:
            Cls_predictions.append(prediction)
        elif ds['labels'][qstate] == 1.0:
            Noncls_predictions.append(prediction)
    Cls_predictions = np.array(Cls_predictions)
    Noncls_predictions = np.array(Noncls_predictions)
    Cls_accuracy = round(1-np.sum(Cls_predictions)/len(Cls_predictions),4)
    Ncl_accuracy = round(np.sum(Noncls_predictions)/len(Noncls_predictions),4)
    return Cls_accuracy, Ncl_accuracy

def Innocenti_witness(ds, epsilon, kshots, modes):
    M = 1000*kshots
    Noncls_predictions = []
    Cls_predictions = []
    count = 0
    for qstate in range(len(ds['labels'])):
        D_ks = []
        x = ds['images'][qstate]
        for j in range(modes):
            n = x[j,:]
            #print(len(n))
            D_k_mode = []
            N = int(np.max(n))
            p_tmp, bin = np.histogram(n, bins = np.arange(int(N+2)), density=True)
            if N == 0:
                D_k_mode.append(0.0)
                D_ks.append(D_k_mode)
            elif N == 1:
                # Filip and Lachmann criterion
                D_k_mode.append(-p_tmp[1]/(p_tmp[0]*np.log(p_tmp[0])) )
                D_ks.append(D_k_mode)
            else:
                Q_k = np.array([scipy.special.factorial(k)*p_tmp[k] for k in range(N+1)])
                partial_sum = np.sum(p_tmp[:N-1])
                
                if p_tmp[N-1] > 1e-6:
                    Q_relation = N*p_tmp[N]/p_tmp[N-1]
                    quotient  = ((p_tmp[N-1]/(N*p_tmp[N]))**N)*Q_k[N]
                    
                    summand = np.array([((Q_relation)**k)/scipy.special.factorial(k) for k in range(N-1)])
                    Q_tail = quotient *(np.exp(Q_relation) - np.sum(summand))
                    tmp_dk = np.add(partial_sum,Q_tail)
                else:
                    print(f'alternative count = {count}') #
                    count += 1
                    tmp_dk = 2.0 # in this case Q_k[N-1] ~ 0 such that the state is non-classical
                D_ks.append(tmp_dk)
            #print(D_ks)
        prediction = float(np.array(D_ks)[0] > (1.0 + epsilon))
        if ds['labels'][qstate] == 0.0:
            Cls_predictions.append(prediction)
        elif ds['labels'][qstate] == 1.0:
            Noncls_predictions.append(prediction)
    Cls_predictions = np.array(Cls_predictions)
    Noncls_predictions = np.array(Noncls_predictions)
    Cls_accuracy = round(1-np.sum(Cls_predictions)/len(Cls_predictions),4)
    Ncl_accuracy = round(np.sum(Noncls_predictions)/len(Noncls_predictions),4)
    return Cls_accuracy, Ncl_accuracy

### Generalized Klyshko (here called Radim-Filip)

This witness is based on standard methods, and considers photon-number probabilities. If $C$ is the largest photon number measured, this method considers $\lbrace p_0, p_1, ..., p_{C-1}\rbrace$ since the last probability $p_C$ is determined by $1-\sum_{i=0}^{C-1}p_i$.

1. #### Integer indexing
We construct a matrix of probabilities, indexed with integer numbers $j,k\in\mathbb{N}$. The so-constructed matrix features even photon-number probabilities on the diagonal $p_0, p_2,...$ and is positive-semidefinite for classical states 
$$ M_{j,k} = \binom{j+k}{j}p_{j+k} \overset{\mathrm{cl.}}{\geq}0 \;;\;\;\; j+k\leq C-1$$


2. #### Half-integer indexing
In order to construct a matrix with odd photon-number probabilities $p_1, p_3, ...$ on the diagonal, we index the matrix elements with half-integers $m,l\in \frac{1}{2}\mathbb{N} \setminus \mathbb{N}$. Analogue to the case of integer indexing, the matrix reads

$$ M_{j,k} = \binom{j+k}{j}p_{j+k} \overset{\mathrm{cl.}}{\geq}0 \;;\;\;\; j+k\leq C-1$$

For real numbers $x,y$ the binomial coefficient is defined as
$$ \binom{x}{y} = \frac{\Gamma(x+1)}{\Gamma(y+1)\Gamma(x-y+1)}$$

In [ ]:
def MinEigenval_int(rho, cutoff):  
    if cutoff < 3:
        return 1.0
    else:
        M = np.zeros((int((cutoff+1)/2),int((cutoff+1)/2)))
        for j in range(int((cutoff+1)/2)):
            for k in range(int((cutoff+1)/2)):
                M[j,k] = scipy.special.binom(j+k,j)*rho[j+k]
        min_eigenval = np.min(linalg.eigvals(M))
        return min_eigenval

def MinEigenval_half_int(rho, cutoff):  
    if cutoff < 4:
        return 1.0
    else:
        M = np.zeros((int(cutoff/2),int(cutoff/2)))
        for j in np.arange(0.5, int(cutoff/2)):
            for k in np.arange(0.5, int(cutoff/2)):
                M[int(j-0.5),int(k-0.5)] = scipy.special.binom(j+k,j)*rho[int(j+k)]
        min_eigenval = np.min(linalg.eigvals(M))
        return min_eigenval

def RadimFilip_witness(ds, tolerance, kshots, modes):
    if modes != 1:
        raise Exception('This code was written for a single mode!')
    list_eigenvals_int = []
    list_eigenvals_half_int = []
    Cls_predictions = []
    Noncls_predictions = []
    for qstate in range(len(ds['labels'])):
        D_ks = []
        x = ds['images'][qstate]
        n = x[0,:]
        max_photon_number = np.max(n)
        p_tmp, bin = np.histogram(n, bins = np.arange(int(max_photon_number+2)), density=True)

        # compute minmal eigenvalue of matrix
        min_eigenval_int = MinEigenval_int(rho=p_tmp, cutoff=max_photon_number)
        min_eigenval_half_int = MinEigenval_half_int(rho=p_tmp, cutoff=max_photon_number)
        list_eigenvals_int.append(min_eigenval_int)
        list_eigenvals_half_int.append(min_eigenval_half_int)

        if ds['labels'][qstate] == 0.0:
            Cls_predictions.append(min(min_eigenval_int, min_eigenval_half_int) <-1.*tolerance)
        elif ds['labels'][qstate] == 1.0:
            Noncls_predictions.append(min(min_eigenval_int, min_eigenval_half_int)<-1.*tolerance)

    # compute accuracy
    Cls_predictions = np.array(Cls_predictions)
    Noncls_predictions = np.array(Noncls_predictions)
    Cls_accuracy = round(1-np.sum(Cls_predictions)/len(Cls_predictions),4)
    Ncl_accuracy = round(np.sum(Noncls_predictions)/len(Noncls_predictions),4)
    return Cls_accuracy, Ncl_accuracy

def Gen_Klyshko(ds, tolerance, kshots, modes):
    list_eigenvals_int = []
    list_eigenvals_half_int = []
    Cls_predictions = []
    Noncls_predictions = []
    for qstate in range(len(ds['labels'])):
        D_ks = []
        x = ds['images'][qstate]
        n = x
        max_photon_number = np.max(n, axis=1)
        p_tmp, bin = np.histogramdd(n.T, bins = [np.arange(int(m+2)) for m in max_photon_number], density=True)
        padded_p = p_tmp
        P = padded_p[:4,:4,:4,:4,:4,:4].flatten()

        # compute minmal eigenvalue of matrix
        min_eigenval_int = MinEigenval_int(rho=P, cutoff=np.max(max_photon_number))
        min_eigenval_half_int = MinEigenval_half_int(rho=P, cutoff=np.max(max_photon_number))
        list_eigenvals_int.append(min_eigenval_int)
        list_eigenvals_half_int.append(min_eigenval_half_int)

        if ds['labels'][qstate] == 0.0:
            Cls_predictions.append(min(min_eigenval_int, min_eigenval_half_int) <-1.*tolerance)
        elif ds['labels'][qstate] == 1.0:
            Noncls_predictions.append(min(min_eigenval_int, min_eigenval_half_int)<-1.*tolerance)
    # compute accuracy
    Cls_predictions = np.array(Cls_predictions)
    Noncls_predictions = np.array(Noncls_predictions)
    Cls_accuracy = round(1-np.sum(Cls_predictions)/len(Cls_predictions),4)
    Ncl_accuracy = round(np.sum(Noncls_predictions)/len(Noncls_predictions),4)
    return Cls_accuracy, Ncl_accuracy

--------------------------------

# Let's use the code

In [ ]:
kshots=1
train_ds = load_h5py_ds('ds16_6modes', kshots=kshots)

print(train_ds['images'].shape) # shape=(num_states, modes, num_samples)
x = train_ds['images'][0]
print('shape of x:', x.shape)


TraditionalPrediction(train_ds, 
                    Gen_Klyshko, 
                    kshots=kshots, 
                    modes=6, 
                    N_detectors=None, 
                    tolerance_min=-2,
                    tolerance_max=30, 
                    step_size=0.001,
                    SAVENAME=f'Gen_Klyshko_ds16_6mode_witness_{kshots}k')

(60, 6, 1000)
shape of x: (6, 1000)


100%|██████████| 33/33 [00:00<00:00, 51.30it/s]
